# Cell 10: Grounding Direction vs Natural Failures
This notebook tests if a detector explicitly trained on the causal 'grounding' direction (clean vs corrupted) can detect natural failures. 
**Requirements:** Attach your `features` dataset (the unzipped `probe_features.npz` files) to this notebook before running.

In [ ]:
!pip install -q huggingface_hub scikit-learn numpy torch

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

print('Downloading hidden states from HuggingFace (this may take a few minutes)...')
hs_local = Path(snapshot_download('AnishRacherla/LinguaFranca-Phase3', repo_type='dataset', allow_patterns='data/hidden_states/*'))
HS_DIR = hs_local / 'data' / 'hidden_states'
print(f'Hidden states ready at {HS_DIR}')

In [ ]:
import json
import os
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from huggingface_hub import hf_hub_download
import joblib

# Find the unzipped numpy arrays in Kaggle
FEATURES_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'ids.npy' in files and 'F_clean.npy' in files:
        FEATURES_DIR = root
        break
if not FEATURES_DIR:
    raise FileNotFoundError("Could not find the unzipped .npy files in /kaggle/input/.")
def load_npy(name):
    return np.load(os.path.join(FEATURES_DIR, name + '.npy'), allow_pickle=True)

REPO = "AnishRacherla/LinguaFranca-Phase3"
LAYERS = [9, 10, 11, 12, 13]
HEDGE = ["no information", "no mention", "not mentioned", "no relevant information", "not found", "cannot find",
         "does not mention", "not provide", "not available", "not specified", "does not specify", "not given",
         "unknown", "cannot be determined", "not stated", "does not contain", "not include", "no details"]

labels_file = hf_hub_download(REPO, "data/2wikimultihopqa/augmented.jsonl", repo_type="dataset")
train_ids = set(map(str, json.load(open(hf_hub_download(REPO, "probes/probe_train_ids.json", repo_type="dataset")))))
examples = [json.loads(l) for l in open(labels_file)]

# ---- natural hop-1 data: probe TRAIN split (for the scaler) and TEST split (for evaluation) ----
nat = {"train": {"X": {L: [] for L in LAYERS}, "y": []}, "test": {"X": {L: [] for L in LAYERS}, "y": [], "hedge": []}}
for ex in examples:
    if ex.get("is_counterfactual", False):
        continue
    split = "train" if str(ex["id"]) in train_ids else "test"
    p = HS_DIR / f"{ex['id']}.pt"
    if not p.exists():
        continue
    pooled = torch.load(p, map_location="cpu", weights_only=False)["pooled"]
    for hop in ex.get("hops", []):
        if hop["hop_idx"] == 1 and hop.get("label") in (0, 1) and pooled.shape[1] > 0:
            for L in LAYERS:
                nat[split]["X"][L].append(pooled[L, 0].float().numpy())
            nat[split]["y"].append(hop["label"])
            if split == "test":
                t = (hop.get("text") or "").lower()
                nat[split]["hedge"].append((not t) or any(k in t for k in HEDGE))
for s in nat:
    nat[s]["X"] = {L: np.nan_to_num(np.stack(v)) for L, v in nat[s]["X"].items()}
    nat[s]["y"] = np.array(nat[s]["y"])
y_te = nat["test"]["y"]; hedge_te = np.array(nat["test"]["hedge"])
print(f"Natural hop-1: train {len(nat['train']['y'])} | test {len(y_te)} "
      f"(failures {y_te.sum()}: hedges {(hedge_te & (y_te == 1)).sum()}, confident {(~hedge_te & (y_te == 1)).sum()})")

# ---- minimal-pair features ----
FL = [int(v) for v in load_npy('feature_layers')]; CL = [int(v) for v in load_npy('control_layers')]
pid = load_npy('ids').astype(str)
tr = np.array([i in train_ids for i in pid]); ho = ~tr
hc = load_npy('has_control')
F_clean = load_npy('F_clean'); F_cf = load_npy('F_cf')
F_entity = load_npy('F_entity'); F_control = load_npy('F_control')
print(f"Minimal pairs: {tr.sum()} for learning the direction, {ho.sum()} held out\n")

def aurocs(score):
    out = [roc_auc_score(y_te, score)]
    for m in (hedge_te, ~hedge_te):
        sel = (y_te == 0) | ((y_te == 1) & m)
        out.append(roc_auc_score(y_te[sel], score[sel]) if (y_te[sel] == 1).any() else np.nan)
    return out

rows = []
for L in LAYERS:
    k = FL.index(L)
    sc = StandardScaler().fit(nat["train"]["X"][L])
    z = lambda A: sc.transform(np.nan_to_num(A.reshape(-1, A.shape[-1]).astype(np.float64))).reshape(A.shape)
    Zc, Zf = z(F_clean[:, k]), z(F_cf[:, k])
    Ze, Zctl = z(F_entity[:, :, k]), z(F_control[:, :, k])

    methods = {}
    v = (Zf[tr] - Zc[tr]).mean(0); v /= max(np.linalg.norm(v), 1e-12)
    methods["diff-of-means"] = lambda A, v=v: A @ v
    lr = LogisticRegression(C=0.01, max_iter=3000).fit(np.r_[Zc[tr], Zf[tr]], np.r_[np.zeros(tr.sum()), np.ones(tr.sum())])
    methods["pair-logreg"] = lambda A, lr=lr: (A.reshape(-1, A.shape[-1]) @ lr.coef_[0]).reshape(A.shape[:-1])

    for name, f in methods.items():
        sc_c, sc_f = f(Zc), f(Zf)
        sep = (sc_f[ho] > sc_c[ho]).mean()
        gap = (sc_f[ho] - sc_c[ho]).mean()
        gap = gap if abs(gap) > 1e-9 else np.nan
        sc_e = f(Ze)
        restore = ((sc_f[ho, None] - sc_e[ho][:, :L]).mean(0) / gap).max()
        cl_ok = [c for c in CL if c < L]
        sel = ho & hc
        ctl = ((sc_f[sel, None] - f(Zctl)[sel][:, [CL.index(c) for c in cl_ok]]).mean(0) / gap).max() if cl_ok and sel.any() else np.nan
        a_all, a_h, a_c = aurocs(f(sc.transform(nat["test"]["X"][L])))
        rows.append((L, name, sep, gap, restore, ctl, a_all, a_h, a_c))

    probe = joblib.load(hf_hub_download(REPO, f"probes/hop1_probe_layer{L}.joblib", repo_type="dataset"))
    rows.append((L, "failure probe", np.nan, np.nan, np.nan, np.nan, *aurocs(probe.decision_function(nat["test"]["X"][L]))))

print(f"{'L':>3} | {'detector':<14} | held-out pairs: sep  gap   restore  ctrl | natural AUROC: all  hedge  confident")
for L, name, sep, gap, rest, ctl, a, h, c in rows:
    pairs = (f"{sep*100:4.0f}% {gap:+5.2f} {rest:+7.2f} {ctl:+6.2f}" if not np.isnan(sep) else f"{'-':>27}")
    print(f"{L:>3} | {name:<14} | {pairs:>31} |  {a:.3f}  {h:.3f}  {c:.3f}")
    if name == "failure probe":
        print("    " + "-" * 92)

json.dump([dict(zip(["layer", "detector", "heldout_sep", "heldout_gap", "restore", "control_restore",
                     "auroc_all", "auroc_hedge", "auroc_confident"], [r[0], r[1], *map(float, r[2:])])) for r in rows],
          open("grounding_direction_results.json", "w"), indent=1)
print("\nSaved grounding_direction_results.json")
